In [1]:
import pandas as pd
import numpy as np
import requests
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
def get_adj_risk_score():
    results = pd.read_csv("cta_results/pair_results.csv")
    results['response'] = np.where((results['response'] == "Student 1") | (results['response'] == "Student 2"), results['response'], "Removed response")

    rating_counts_df = pd.concat([
    results.loc[results['response'] == 'Student 1', 'id.x'],
    results.loc[results['response'] == 'Student 2', 'id.y']
    ])

    rate_df = rating_counts_df.value_counts().reset_index()
    rate_df.columns = ['id', 'risk_score']

    unknown_counts_df = pd.concat([
    results.loc[results['response'] == 'Removed response', 'id.x'],
    results.loc[results['response'] == 'Removed response', 'id.y']
    ])

    unknown_df = unknown_counts_df.value_counts().reset_index()
    unknown_df.columns = ['id', 'unknown_count']

    cta = pd.read_csv("cta_results/cta_preprocessed_unpaired.csv")
    results_df = pd.merge(pd.merge(cta, rate_df, on = 'id', how = 'left'), unknown_df, on = 'id', how = 'left')
    results_df['risk_score'] = results_df['risk_score'].fillna(0)
    results_df['unknown_count'] = results_df['unknown_count'].fillna(0)
    results_df['risk_score_adj'] = results_df['risk_score']/(results_df['pair_group_size'] - results_df['unknown_count'])

    return results_df

In [3]:
results = get_adj_risk_score()
cta = pd.read_csv("cta_student_level_final.csv")
cta = cta[['state', 'grdlvl', 'race', 'sex', 'spec_speced', 'spec_gifted', 'spec_esl', 'frl', 'y_yirt' ,'xirt', 'treatment']]
cta['id'] = range(1, len(cta) + 1)

results = results[['id', 'oob_preds', 'pair_group', 'pair_group_size', 'risk_score', 'risk_score_adj']]

merged_results = results.merge(cta, on = 'id', how = 'left')

In [4]:
# preprocessing for model 

# for categorical variables, replace NAs with "unknown"
merged_results['race'] = merged_results['race'].fillna("Unknown")
merged_results['sex'] = merged_results['sex'].fillna("Unknown")
merged_results['spec_speced'] = merged_results['spec_speced'].fillna("Unknown")
merged_results['spec_gifted'] = merged_results['spec_gifted'].fillna("Unknown")
merged_results['spec_esl'] = merged_results['spec_esl'].fillna("Unknown")
merged_results['frl'] = merged_results['frl'].fillna("Unknown")

# for numeric, mean impute and add another column that captures the missing values 
merged_results["xirt_mis"] = merged_results["xirt"].isna().astype(int)
merged_results['xirt'] = merged_results['xirt'].fillna(merged_results['xirt'].mean())

X = merged_results[["state", "grdlvl", 'race', 'sex', 'spec_speced', 'spec_gifted', 'spec_esl', 'frl', 'xirt_mis', 'oob_preds', 'xirt', 'y_yirt', 'risk_score_adj', 'treatment']]
df_dummies = pd.get_dummies(X, columns=["state", "grdlvl", 'race', 'sex', 'spec_speced', 'spec_gifted', 'spec_esl', 'frl', 'xirt_mis'], drop_first=True)

In [5]:
import statsmodels.api as sm

#note: takes about 20 minutes to run 
y = df_dummies["y_yirt"]


# first without risk score 
X1 = df_dummies.drop(['y_yirt', 'risk_score_adj'], axis = 1).astype(float)
X1= sm.add_constant(X1)

X2 = df_dummies.drop(['y_yirt'], axis = 1).astype(float)
X2= sm.add_constant(X2)

for i in df_dummies.index:
    X_withouti = sm.add_constant(X1.drop(index = i))
    y_withouti = np.delete(y, i)
    model_withouti = sm.OLS(y_withouti, X_withouti).fit()

    observation_it = X1.iloc[[i]]
    observation_it.loc[:,'treatment'] = 1 

    observation_ic = X1.iloc[[i]]
    observation_ic.loc[:,'treatment'] = 0

    y_it = model_withouti.predict(observation_it).iloc[0]
    y_ic = model_withouti.predict(observation_ic).iloc[0]
    
    df_dummies.loc[i, 'y_ic_basic'] = y_ic
    df_dummies.loc[i, 'y_it_basic'] = y_it

    X_withouti = sm.add_constant(X2.drop(index = i))
    y_withouti = np.delete(y, i)
    model_withouti = sm.OLS(y_withouti, X_withouti).fit()

    observation_it = X2.iloc[[i]]
    observation_it.loc[:,'treatment'] = 1 

    observation_ic = X2.iloc[[i]]
    observation_ic.loc[:,'treatment'] = 0

    y_it = model_withouti.predict(observation_it).iloc[0]
    y_ic = model_withouti.predict(observation_ic).iloc[0]
    
    df_dummies.loc[i, 'y_ic_score'] = y_ic
    df_dummies.loc[i, 'y_it_score'] = y_it


In [6]:
def get_variance(df, p, pred_type = "basic", outcome = "y_yirt", treatment = "treatment"):
    df['m'] = p * df[f'y_it_{pred_type}'] + (1- p) * df[f'y_ic_{pred_type}']

    nc = len(df) - sum(df[f'{treatment}'])
    nt = sum(df[f'{treatment}'])
    N = len(df)

    ec2 = 1/nc * sum((1 - df[f'{treatment}']) * (df[f'y_ic_{pred_type}'] - df[f'{outcome}'])**2)
    et2 = 1/nt * sum(df[f'{treatment}'] * (df[f'y_it_{pred_type}'] - df[f'{outcome}'])**2)

    var_est = 1/N * (p/(1-p) * ec2 + (1-p)/p * et2 + 2 * (ec2 * et2)**0.5)
    return var_est

In [7]:
var_without = get_variance(df_dummies, p = 0.5) # variance without risk score
var_with = get_variance(df_dummies, pred_type = "score", p = 0.5) # variance with risk score 
print(var_without**0.5, var_with**0.5)

0.009577002926348924 0.009571130241367748
